# Environments: implicit solvent and QM/MM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mgoldey/ferric/blob/main/examples/notebooks/03_solvent_and_qmmm.ipynb)

A water molecule in a continuum solvent (IEF-PCM), and in the field of a second water treated as point charges (QM/MM electrostatic embedding).

Runs on a Colab CPU runtime in about two minutes. Every number printed below is also asserted, so a cell that runs without an error has reproduced the reference value.

In [ ]:
%pip install -q --pre ferric py3Dmol
import os

os.environ.setdefault(
    "OPENBLAS_NUM_THREADS", "1"
)  # set before the first `import ferric`

In [ ]:
import numpy as np
import ferric

try:
    from ferric import viz
except ImportError as err:
    raise ImportError(
        "this notebook needs a ferric release that ships ferric.viz "
        "(newer than 0.1.0rc11)"
    ) from err

## Implicit solvent: IEF-PCM

`solvent=` takes a name (`"water"`) or a dielectric constant. The certificate lists the approximation. Energies are in Hartree; 1 Ha = 627.5095 kcal/mol.

In [ ]:
WATER_XYZ = """3
water
O   0.000000   0.000000   0.117790
H   0.000000   0.755453  -0.471161
H   0.000000  -0.755453  -0.471161
"""
water = ferric.Molecule.from_xyz_string(WATER_XYZ, 0, 1)  # charge 0, singlet
KCAL = 627.5095
bs = ferric.BasisSet.bundled("cc-pvdz")
gas = ferric.run_rhf(water, bs)
pcm = ferric.run_rhf(water, bs, solvent="water")
assert gas.converged and pcm.converged
dg = (pcm.energy - gas.energy) * KCAL
print(f"gas     {gas.energy:.8f} Ha")
print(
    f"in water{pcm.energy:.8f} Ha   electrostatic solvation energy {dg:+.2f} kcal/mol"
)
print([a["name"] + ": " + a["detail"] for a in pcm.certificate["approximations"]])

# Stabilisation by a polar solvent; magnitude measured with this build (HF/cc-pVDZ, IEF-PCM, no
# cavitation or dispersion terms, so it is the electrostatic part only).
assert -7.0 < dg < -5.5

The stabilisation grows with the dielectric constant and saturates, as the reaction field does:

In [ ]:
eps_values = [2.0, 4.8, 20.0, 78.39]
dgs = []
for eps in eps_values:
    r = ferric.run_rhf(water, bs, solvent=eps)
    assert r.converged
    dgs.append((r.energy - gas.energy) * KCAL)
    print(f"epsilon = {eps:6.2f}   dE = {dgs[-1]:+.2f} kcal/mol")
assert all(a > b for a, b in zip(dgs, dgs[1:]))  # more negative with larger epsilon

## QM/MM: a water dimer

The full structure goes in (Angstrom) with MM point charges (e) for every atom; `qm_indices` chooses the quantum region and the rest become point charges that polarise it. Charges of QM atoms are ignored. The MM water uses TIP3P charges.

In [ ]:
symbols = ["O", "H", "H", "O", "H", "H"]
coords = [
    [-1.551007, -0.114520, 0.000000],
    [-1.934259, 0.762503, 0.000000],
    [-0.599677, 0.040712, 0.000000],
    [1.350625, 0.111469, 0.000000],
    [1.680398, -0.373741, -0.758561],
    [1.680398, -0.373741, 0.758561],
]
charges = [-0.834, 0.417, 0.417] * 2

system = ferric.QmmmSystem(symbols, coords, charges, qm_indices=[0, 1, 2])
qmmm = ferric.run_qmmm(system, "cc-pvdz", method="rhf")
print(f"QM/MM energy {qmmm.energy:.8f} Ha")


def xyz(idx):
    rows = [f"{symbols[i]} {coords[i][0]} {coords[i][1]} {coords[i][2]}" for i in idx]
    return f"{len(idx)}\n\n" + "\n".join(rows)


def mol(idx):
    return ferric.Molecule.from_xyz_string(xyz(idx))


qm_alone = ferric.run_rhf(mol([0, 1, 2]), bs)
embedding = (qmmm.energy - qm_alone.energy) * KCAL
print(
    f"embedding energy of the QM water in the MM water's field: {embedding:+.2f} kcal/mol"
)

For comparison, the same dimer fully quantum mechanical. The two numbers are not the same quantity: the QM/MM figure has no exchange repulsion, no charge transfer and no polarisation of the MM water, and TIP3P charges are a different description of the second molecule. The agreement to about a kcal/mol shows what electrostatic embedding does capture.

In [ ]:
dimer = ferric.run_rhf(mol(range(6)), bs)
mm_water = ferric.run_rhf(mol([3, 4, 5]), bs)
full_qm = (dimer.energy - qm_alone.energy - mm_water.energy) * KCAL
print(
    f"full QM interaction energy (HF/cc-pVDZ, no counterpoise): {full_qm:+.2f} kcal/mol"
)
assert qmmm.converged and dimer.converged
assert embedding < 0 and abs(embedding - full_qm) < 2.0  # measured gap 0.9 kcal/mol

grad = np.array(qmmm.qm_gradient())
print("QM-atom gradient dE/dR (Ha/Bohr), shape", grad.shape)
print("largest component:", f"{np.abs(grad).max():.4f}")
assert grad.shape == (3, 3) and np.isfinite(grad).all()

In [ ]:
viz.show_molecule(mol(range(6)))